<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-12-production-deploy/lesson-12.2-rag-backend/notebooks/GCP_Capstone_12.2_RAGBackend.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12.2 RAG API Backend (FastAPI on Cloud Run)
**Netsetos GenAI Engineering — GCP Capstone**

FastAPI service that answers RAG queries: embed the query, ANN search Vector Search, rerank, generate with citations, stream via Server-Sent Events. Pydantic-typed, traced, under 500 LOC.

## Cell 1: requirements.txt (pinned April 2026)

In [ ]:
REQUIREMENTS = '''\
fastapi==0.115.6
uvicorn[standard]==0.32.1
gunicorn==23.0.0
pydantic==2.10.4
pydantic-settings==2.7.0
google-cloud-aiplatform==1.95.0
google-cloud-firestore==2.20.0
google-genai==2.20.0
google-cloud-discoveryengine>=0.13.0
vertexai==1.95.0
google-cloud-logging==3.11.3
opentelemetry-api==1.30.0
opentelemetry-sdk==1.30.0
opentelemetry-exporter-gcp-trace==1.9.0
opentelemetry-instrumentation-fastapi==0.51b0
tenacity==9.0.0
httpx==0.28.1
tiktoken==0.9.0
'''
with open('requirements.txt', 'w') as f: f.write(REQUIREMENTS)
print('requirements.txt written')

## Cell 2: config.py — Pydantic settings

In [ ]:
CONFIG_PY = '''
from pydantic import Field
from pydantic_settings import BaseSettings, SettingsConfigDict

class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env", extra="ignore")

    project_id: str = Field(alias="GOOGLE_CLOUD_PROJECT")
    region: str = "us-central1"
    india_region: str = "asia-south1"
    vector_index_endpoint: str = Field(alias="VECTOR_INDEX_ENDPOINT")
    vector_deployed_index: str = Field(alias="VECTOR_DEPLOYED_INDEX_ID")
    embed_model: str = "text-embedding-005"
    generator_model: str = "gemini-3.6-flash"
    rerank_model: str = "semantic-ranker-fast-004"
    top_k_retrieve: int = 20
    top_k_rerank: int = 5
    max_context_tokens: int = 8000
    max_answer_tokens: int = 1024

settings = Settings()
'''
with open('config.py', 'w') as f: f.write(CONFIG_PY)
print('config.py written')

## Cell 3: schemas.py — request / response Pydantic models

In [ ]:
SCHEMAS_PY = '''
from typing import Literal, Optional, List
from pydantic import BaseModel, Field

class QueryRequest(BaseModel):
    query: str = Field(min_length=1, max_length=4000)
    tenant_id: str = Field(min_length=1)
    user_id: str = Field(min_length=1)
    top_k: int = Field(default=5, ge=1, le=20)
    stream: bool = True
    filters: Optional[dict] = None  # e.g. {"doc_type": "policy"}

class Citation(BaseModel):
    chunk_id: str
    source_uri: str
    page: Optional[int] = None
    quote: str = Field(max_length=500)
    score: float = Field(ge=0, le=1)

class RAGAnswer(BaseModel):
    answer: str
    citations: List[Citation]
    confidence: Literal["high", "medium", "low"]
    answerable: bool
    model: str
    tokens_in: int
    tokens_out: int
    latency_ms: int

class StreamEvent(BaseModel):
    # Server-Sent Events payload
    event: Literal["token", "citation", "done", "error"]
    data: dict
'''
with open('schemas.py', 'w') as f: f.write(SCHEMAS_PY)
print('schemas.py written')

## Cell 4: retriever.py — embed + ANN search + rerank

In [ ]:
RETRIEVER_PY = '''
from functools import lru_cache
from google.cloud import aiplatform
from google.cloud import discoveryengine_v1 as discoveryengine
from google import genai
from google.genai import types
from google.cloud import firestore
from config import settings

@lru_cache(maxsize=1)
def _genai_client():
    return genai.Client(enterprise=True, project=settings.project_id, location=settings.region)

@lru_cache(maxsize=1)
def _index_endpoint():
    return aiplatform.MatchingEngineIndexEndpoint(settings.vector_index_endpoint)

@lru_cache(maxsize=1)
def _fs():
    return firestore.Client(project=settings.project_id, database="(default)")

def embed_query(q: str) -> list[float]:
    resp = _genai_client().models.embed_content(
        model=settings.embed_model, contents=q,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY", output_dimensionality=768))
    return resp.embeddings[0].values

def retrieve(query: str, tenant_id: str, top_k: int, filters: dict | None = None) -> list[dict]:
    vec = embed_query(query)
    restricts = [{"namespace": "tenant_id", "allow": [tenant_id]}]
    if filters:
        for k, v in filters.items():
            restricts.append({"namespace": k, "allow": [str(v)]})
    resp = _index_endpoint().find_neighbors(
        deployed_index_id=settings.vector_deployed_index,
        queries=[vec], num_neighbors=settings.top_k_retrieve,
        filter=restricts,
    )
    ids = [n.id for n in resp[0]]
    scores = {n.id: n.distance for n in resp[0]}
    # Fan-out to Firestore for chunk payloads
    chunks = []
    for doc in _fs().collection("chunks").where("__name__", "in", ids[:10]).stream():
        d = doc.to_dict(); d["id"] = doc.id; d["score"] = scores.get(doc.id, 0)
        chunks.append(d)
    return chunks

@lru_cache(maxsize=1)
def _ranker():
    return discoveryengine.RankServiceClient()

def rerank(query: str, chunks: list[dict], k: int) -> list[dict]:
    if not chunks: return chunks
    client = _ranker()
    ranking_config = client.ranking_config_path(
        project=settings.project_id, location="global",
        ranking_config="default_ranking_config")
    records = [discoveryengine.RankingRecord(id=str(i), content=c["text"])
               for i, c in enumerate(chunks)]
    resp = client.rank(request=discoveryengine.RankRequest(
        ranking_config=ranking_config,
        model=settings.rerank_model,
        top_n=k, query=query, records=records))
    out = []
    for r in resp.records:
        chunks[int(r.id)]["rerank_score"] = r.score
        out.append(chunks[int(r.id)])
    return out
'''
with open('retriever.py', 'w') as f: f.write(RETRIEVER_PY)
print('retriever.py written')

## Cell 5: generator.py — structured answer with citations

In [ ]:
GENERATOR_PY = '''
from google import genai
from google.genai import types
from schemas import RAGAnswer, Citation
from config import settings

_client = genai.Client(enterprise=True, project=settings.project_id, location="global")  # generation runs on the global endpoint (Gemini 3.x)

SYSTEM = """You are DocuMind, a retrieval-grounded assistant.
Rules:
1. Answer ONLY from the numbered context below. Never invent sources.
2. Cite using [N] where N is the chunk number. Multiple chunks: [1,2].
3. If the context does not contain the answer, set answerable=false and say so.
4. Keep answers under 300 words unless asked for more.
"""

def build_context(chunks: list[dict]) -> str:
    lines = []
    for i, c in enumerate(chunks, 1):
        src = c.get("source_uri", "")
        page = c.get("page_start")
        header = f"[{i}] {src}" + (f" page {page}" if page else "")
        lines.append(f"{header}\\n{c['text']}")
    return "\\n\\n".join(lines)

def generate(query: str, chunks: list[dict]) -> RAGAnswer:
    context = build_context(chunks)
    prompt = f"{SYSTEM}\\n\\nContext:\\n{context}\\n\\nQuestion: {query}"

    r = _client.models.generate_content(
        model=settings.generator_model,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema={
                "type": "OBJECT",
                "properties": {
                    "answer":     {"type": "STRING"},
                    "citations":  {"type": "ARRAY", "items": {"type": "INTEGER"}},
                    "confidence": {"type": "STRING", "enum": ["high","medium","low"]},
                    "answerable": {"type": "BOOLEAN"},
                },
                "required": ["answer","citations","confidence","answerable"],
            },
            temperature=0.1,
            max_output_tokens=settings.max_answer_tokens,
            thinking_config=types.ThinkingConfig(thinking_budget=0),
        ),
    )
    parsed = r.parsed or {}
    cits = []
    for idx in parsed.get("citations", []):
        if 1 <= idx <= len(chunks):
            c = chunks[idx - 1]
            cits.append(Citation(chunk_id=c["id"], source_uri=c["source_uri"],
                                 page=c.get("page_start"), quote=c["text"][:500],
                                 score=float(c.get("rerank_score") or c.get("score") or 0)))
    return RAGAnswer(
        answer=parsed.get("answer", ""),
        citations=cits,
        confidence=parsed.get("confidence", "low"),
        answerable=bool(parsed.get("answerable", False)),
        model=settings.generator_model,
        tokens_in=r.usage_metadata.prompt_token_count or 0,
        tokens_out=r.usage_metadata.candidates_token_count or 0,
        latency_ms=0,
    )
'''
with open('generator.py', 'w') as f: f.write(GENERATOR_PY)
print('generator.py written')

## Cell 6: main.py — FastAPI app with /query, /stream, /health

In [ ]:
MAIN_PY = '''
import time, json, logging
from fastapi import FastAPI, Request, HTTPException, Depends
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from opentelemetry import trace
from opentelemetry.instrumentation.fastapi import FastAPIInstrumentor
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.cloud_trace import CloudTraceSpanExporter
from schemas import QueryRequest, RAGAnswer
from retriever import retrieve, rerank
from generator import generate
from config import settings

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s")
log = logging.getLogger("documind-api")

trace.set_tracer_provider(TracerProvider())
trace.get_tracer_provider().add_span_processor(
    BatchSpanProcessor(CloudTraceSpanExporter(project_id=settings.project_id)))
tracer = trace.get_tracer(__name__)

app = FastAPI(title="DocuMind API", version="1.0.0")
FastAPIInstrumentor.instrument_app(app)

app.add_middleware(CORSMiddleware,
    allow_origins=["https://documind.example.com"],
    allow_methods=["POST","GET"], allow_headers=["*"])

def verify_iap(request: Request) -> dict:
    # In production, verify the signed X-Goog-IAP-JWT-Assertion header (see 12.4 auth.py).
    # For tests, allow X-User headers.
    email = request.headers.get("x-user-email")
    tenant = request.headers.get("x-tenant-id")
    if not email or not tenant:
        raise HTTPException(401, "missing identity headers")
    return {"email": email, "tenant_id": tenant}

@app.get("/health")
def health(): return {"status": "ok"}

@app.get("/ready")
def ready():
    # Lazy-init resource probes keep cold start fast; only warm when ready is probed
    from retriever import _embed_model, _index_endpoint, _fs
    _ = _embed_model(); _ = _index_endpoint(); _ = _fs()
    return {"status": "ready"}

@app.post("/v1/query", response_model=RAGAnswer)
def query(req: QueryRequest, user=Depends(verify_iap)):
    if req.tenant_id != user["tenant_id"]:
        raise HTTPException(403, "tenant mismatch")
    t0 = time.time()
    with tracer.start_as_current_span("retrieve"):
        chunks = retrieve(req.query, req.tenant_id, req.top_k, req.filters)
    with tracer.start_as_current_span("rerank"):
        chunks = rerank(req.query, chunks, req.top_k)
    with tracer.start_as_current_span("generate"):
        ans = generate(req.query, chunks)
    ans.latency_ms = int((time.time() - t0) * 1000)
    log.info(json.dumps({"event":"query","tenant":req.tenant_id,"user":user["email"],
                        "latency_ms":ans.latency_ms,"tokens_in":ans.tokens_in,
                        "tokens_out":ans.tokens_out,"answerable":ans.answerable,
                        "confidence":ans.confidence}))
    return ans

@app.post("/v1/stream")
def stream(req: QueryRequest, user=Depends(verify_iap)):
    if req.tenant_id != user["tenant_id"]:
        raise HTTPException(403, "tenant mismatch")
    def sse():
        chunks = retrieve(req.query, req.tenant_id, req.top_k, req.filters)
        chunks = rerank(req.query, chunks, req.top_k)
        for i, c in enumerate(chunks, 1):
            yield f"event: citation\\ndata: {json.dumps({'n': i, 'source': c['source_uri'], 'page': c.get('page_start')})}\\n\\n"
        ans = generate(req.query, chunks)
        for tok in ans.answer.split():
            yield f"event: token\\ndata: {json.dumps({'t': tok + ' '})}\\n\\n"
        yield f"event: done\\ndata: {ans.model_dump_json()}\\n\\n"
    return StreamingResponse(sse(), media_type="text/event-stream")
'''
with open('main.py', 'w') as f: f.write(MAIN_PY)
print('main.py written')

## Cell 7: Dockerfile + Cloud Run deploy

In [ ]:
DOCKERFILE = '''
FROM python:3.12-slim
RUN apt-get update && apt-get install -y --no-install-recommends tini ca-certificates && rm -rf /var/lib/apt/lists/*
RUN useradd --create-home --shell /bin/bash --uid 10001 app
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY --chown=app:app . .
USER app
ENV PORT=8080 PYTHONUNBUFFERED=1
EXPOSE 8080
HEALTHCHECK --interval=20s --timeout=3s --start-period=15s CMD python -c "import urllib.request as r; r.urlopen('http://localhost:8080/health').read()" || exit 1
ENTRYPOINT ["/usr/bin/tini", "--"]
CMD ["gunicorn", "-k", "uvicorn.workers.UvicornWorker", "-w", "2", "-b", "0.0.0.0:8080", "-t", "120", "--access-logfile", "-", "main:app"]
'''
with open('Dockerfile', 'w') as f: f.write(DOCKERFILE)

DEPLOY = '''
gcloud run deploy documind-api \\
  --image=us-central1-docker.pkg.dev/$PROJECT/documind/api:$GIT_SHA \\
  --region=us-central1 --platform=managed \\
  --no-allow-unauthenticated \\
  --ingress=internal-and-cloud-load-balancing \\
  --memory=1Gi --cpu=2 --concurrency=40 --timeout=120 \\
  --min-instances=0 --max-instances=20 \\
  --cpu-boost --execution-environment=gen2 \\
  --service-account=documind-api-sa@$PROJECT.iam.gserviceaccount.com \\
  --set-env-vars="VECTOR_INDEX_ENDPOINT=projects/$PROJECT/.../indexEndpoints/$EID,VECTOR_DEPLOYED_INDEX_ID=dep_001" \\
  --vpc-connector=projects/$PROJECT/locations/us-central1/connectors/documind-vpc \\
  --vpc-egress=private-ranges-only
'''
print(DEPLOY)

## Cell 8: Smoke tests + what the UI inherits

In [ ]:
SMOKE = '''
# Get an identity token (not your user token) because --no-allow-unauthenticated
TOK=$(gcloud auth print-identity-token --impersonate-service-account=documind-ui-sa@$PROJECT.iam.gserviceaccount.com)

curl -sSf -H "Authorization: Bearer $TOK" https://documind-api-xxx.run.app/health
# -> {"status":"ok"}

curl -sSf -H "Authorization: Bearer $TOK" \\
     -H "X-User-Email: alice@acme.in" -H "X-Tenant-Id: tenant-acme" \\
     -H "Content-Type: application/json" \\
     -d '{"query":"Which file types are supported?","tenant_id":"tenant-acme","user_id":"u_1","top_k":5,"stream":false}' \\
     https://documind-api-xxx.run.app/v1/query
# -> {"answer":"...","citations":[...],"confidence":"high","answerable":true,...}

curl -sN -H "Authorization: Bearer $TOK" \\
     -H "X-User-Email: alice@acme.in" -H "X-Tenant-Id: tenant-acme" \\
     -H "Content-Type: application/json" \\
     -d '{"query":"Same","tenant_id":"tenant-acme","user_id":"u_1","top_k":5}' \\
     https://documind-api-xxx.run.app/v1/stream
# -> event: citation ... event: token ... event: done
'''
print(SMOKE)
print()
INHERITS = {
    '12.3 Admin Dashboard': 'structured JSON logs (query/tenant/tokens) feed BigQuery via Log Sink',
    '12.4 Streamlit UI': 'POST /v1/stream via Server-Sent Events; citations array drives pill rendering',
    'Observability': 'Every span traced: retrieve / rerank / generate. OpenTelemetry -> Cloud Trace',
    'Tenant isolation': 'Vector Search restrict on tenant_id + server-side verify tenant matches IAP claim',
}
print('WHAT DOWNSTREAM INHERITS:')
for k, v in INHERITS.items(): print(f'  {k:28} <- {v}')

## ✅ Lesson 12.2 Complete!

- ✅ FastAPI app, Pydantic request/response schemas, OpenTelemetry → Cloud Trace
- ✅ Embed (RETRIEVAL_QUERY) + ANN via MatchingEngineIndexEndpoint with tenant restrict
- ✅ Firestore chunk payload fan-out; semantic rerank via the Vertex AI Ranking API
- ✅ Structured generation with JSON schema (answer / citations / confidence / answerable)
- ✅ `/v1/query` for JSON, `/v1/stream` for Server-Sent Events (citations → tokens → done)
- ✅ `/health` (cheap) + `/ready` (warms caches) for Cloud Run probes
- ✅ Tenant boundary enforced: request `tenant_id` must match IAP-verified claim
- ✅ Gunicorn + UvicornWorker + tini, deployed with `--no-allow-unauthenticated` + private-ranges VPC egress

**Next: Lesson 12.3 — Admin Dashboard &amp; Observability**